In [1]:
import os, json, re
import asyncio
from typing import List, Dict, Any, Optional, Literal
from datetime import datetime
from IPython.display import Markdown, display

from pydantic import BaseModel, Field, ValidationError

from agents import Agent, Runner, Trace, ModelSettings, function_tool
from agents.mcp import MCPServerStdio

from dotenv import load_dotenv

try:
    import pyodbc
except Exception:
    pyodbc = None

load_dotenv()

True

In [2]:
# Global runtime config (keep small and practical)
DEFAULT_AGENT_TEMPERATURE = 0.5
SQL_ANALYST_TEMPERATURE = 0.1
MCP_CLIENT_TIMEOUT_SECONDS = 60
MODEL_PLANNER = "gpt-4o-mini"
MODEL_GENERAL = "gpt-5.2"
SMALL_ROW_THRESHOLD = 1000
MAX_DIRECT_COLUMNS = 30
KNOWLEDGE_BASE_DIR = os.getenv("KNOWLEDGE_BASE_DIR") or "knowledge_base"
SHARED_DOCS_SUBDIR = "_shared"
MAX_TURNS_PLANNER = 12
MAX_TURNS_SQL_ANALYST_DISCOVER = 12
MAX_TURNS_SQL_ANALYST_QUERY = 30
MAX_TURNS_CHART_MAPPER = 12
MAX_TURNS_NARRATOR = 12
MAX_TURNS_CHAT_ORCHESTRATOR = 10
KB_ALLOWED_EXTENSIONS = {".pdf", ".doc", ".docx", ".xlsx", ".xls", ".xlsm"}
MAX_KB_CANDIDATE_FILES = 30

In [3]:

# =========================
# MCP SERVERS + RUNTIME SERVICES
# =========================

# MSSQL MCP
sql_params = {
    "command": "mssql-mcp-server",
    "env": {"MSSQL_CONNECTION_STRING": os.getenv("MSSQL_CONNECTION_STRING")},
}

# Chart MCP (optional, used only when a chart is clearly helpful)
chart_params = {"command": "npx", "args": ["-y", "@antv/mcp-server-chart"]}

# PDF Reader MCP (optional knowledge base context)
pdf_reader_params = {"command": "npx", "args": ["-y", "@sylphx/pdf-reader-mcp"]}
# Word Reader MCP (optional knowledge base context)
word_reader_params = {"command": "uvx", "args": ["--from", "office-word-mcp-server", "word_mcp_server"]}
# Excel MCP (optional knowledge base context)
excel_params = {"command": "uvx", "args": ["excel-mcp-server", "stdio"]}

mssql_server = MCPServerStdio(params=sql_params, client_session_timeout_seconds=MCP_CLIENT_TIMEOUT_SECONDS)
chart_server = MCPServerStdio(params=chart_params, client_session_timeout_seconds=MCP_CLIENT_TIMEOUT_SECONDS)
pdf_reader_server = MCPServerStdio(params=pdf_reader_params, client_session_timeout_seconds=MCP_CLIENT_TIMEOUT_SECONDS)
word_reader_server = MCPServerStdio(params=word_reader_params, client_session_timeout_seconds=MCP_CLIENT_TIMEOUT_SECONDS)
excel_server = MCPServerStdio(params=excel_params, client_session_timeout_seconds=MCP_CLIENT_TIMEOUT_SECONDS)

# Deterministic local memory store (Python-managed, not agent-managed)
# Prefer an explicit writable path; default to /tmp to avoid readonly project mounts.
CHROMA_DIR = os.getenv("CHROMA_DIR") or "/tmp/chroma_ai_analytics"
os.makedirs(CHROMA_DIR, exist_ok=True)

# Local knowledge base folders (PDF docs)
os.makedirs(KNOWLEDGE_BASE_DIR, exist_ok=True)
os.makedirs(os.path.join(KNOWLEDGE_BASE_DIR, SHARED_DOCS_SUBDIR), exist_ok=True)


In [4]:
print("mssql_server")
async with mssql_server:
    tools = await mssql_server.list_tools()
    print([t.name for t in tools])

print("pdf_reader_server")
async with pdf_reader_server:
    tools = await pdf_reader_server.list_tools()
    print([t.name for t in tools])

print("word_reader_server")
async with word_reader_server:
    tools = await word_reader_server.list_tools()
    print([t.name for t in tools])

print("excel_server")
async with excel_server:
    tools = await excel_server.list_tools()
    print([t.name for t in tools])

mssql_server
['list_connections', 'test_connection', 'list_databases', 'list_tables', 'describe_table', 'sample_data', 'execute_query', 'get_relationships', 'list_stored_procedures', 'describe_stored_procedure', 'list_views', 'describe_view', 'list_triggers', 'describe_trigger', 'list_functions', 'list_indexes', 'analyze_table_stats', 'find_missing_indexes', 'analyze_index_usage', 'analyze_database_size', 'list_constraints', 'analyze_check_constraints', 'list_user_defined_types', 'find_computed_columns', 'list_default_constraints', 'analyze_data_distribution', 'find_lookup_tables', 'analyze_null_patterns', 'detect_audit_columns', 'get_stored_procedure_definition', 'get_multiple_stored_procedure_definitions', 'get_all_stored_procedure_definitions', 'search_stored_procedures_by_content']
pdf_reader_server
['read_pdf']
word_reader_server
['create_document', 'copy_document', 'get_document_info', 'get_document_text', 'get_document_outline', 'list_available_documents', 'get_document_xml', 'i

In [5]:

# =========================
# PYDANTIC OUTPUTS
# =========================

class PlannerDecision(BaseModel):
    require_schema_refresh: bool = True
    rationale: str = ""


class CandidateTable(BaseModel):
    name: str
    confidence: float = Field(ge=0, le=1)
    why: str


class ColumnInfo(BaseModel):
    col: str
    type: Optional[str] = None


class TableColumnsEntry(BaseModel):
    table: str
    columns: List[ColumnInfo] = Field(default_factory=list)


class EavHintLite(BaseModel):
    table: str
    attribute_cols: List[str] = Field(default_factory=list)
    value_cols: List[str] = Field(default_factory=list)


class DBCatalogLite(BaseModel):
    candidate_fact_tables: List[CandidateTable] = Field(default_factory=list)
    table_columns: List[TableColumnsEntry] = Field(default_factory=list)
    eav_hints: List[EavHintLite] = Field(default_factory=list)
    notes: List[str] = Field(default_factory=list)


JsonCell = str | int | float | bool | None


class AnswerTable(BaseModel):
    name: str
    columns: List[str]
    rows: List[List[JsonCell]]


class SQLQuery(BaseModel):
    name: str
    sql: str


class AnalyzeResult(BaseModel):
    mode: Literal["SMALL_DIRECT", "LARGE_SPLIT"]
    count_sql: str
    total_rows: int = Field(ge=0)
    insight_tables: List[AnswerTable] = Field(default_factory=list)
    assumptions: List[str] = Field(default_factory=list)
    caveats: List[str] = Field(default_factory=list)
    sql_appendix: List[SQLQuery] = Field(default_factory=list)
    detail_query: Optional[SQLQuery] = None


class ChartMapping(BaseModel):
    applicable: bool
    chart_type: Literal["line", "column", "bar", "pie"] = "line"
    x_field: Optional[str] = None
    y_field: Optional[str] = None
    category_field: Optional[str] = None
    value_field: Optional[str] = None
    reason: str = ""


class NarratorOutput(BaseModel):
    user_output: str
    developer_output: str


class NotebookRunResult(BaseModel):
    final_output: str
    user_output: Optional[str] = None
    developer_output: Optional[str] = None
    verifier_verdict: Literal["PASS", "FAIL"]
    verification_reasons: List[str] = Field(default_factory=list)


In [6]:

# =========================
# HELPERS: JSON, MEMORY, SQL SAFETY, EXPORT
# =========================

import csv
import tempfile
import ast
import chromadb
from chromadb.utils import embedding_functions


def collection_name_for(db_key: str) -> str:
    safe = re.sub(r"[^a-zA-Z0-9_\-]", "_", db_key)[:60]
    return f"ai_analytics__{safe}"


def kb_safe_key(db_key: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_\-]", "_", db_key)[:80]


def get_kb_paths(db_key: str) -> Dict[str, str]:
    base_dir = os.path.abspath(KNOWLEDGE_BASE_DIR)
    shared_dir = os.path.join(base_dir, SHARED_DOCS_SUBDIR)
    db_dir = os.path.join(base_dir, kb_safe_key(db_key))
    os.makedirs(shared_dir, exist_ok=True)
    os.makedirs(db_dir, exist_ok=True)
    return {
        "knowledge_base_dir": base_dir,
        "knowledge_base_shared_dir": shared_dir,
        "knowledge_base_db_dir": db_dir,
    }


def _tokenize_for_match(text: str) -> List[str]:
    return [t for t in re.findall(r"[a-zA-Z0-9_]+", (text or "").lower()) if len(t) >= 3]


def _iter_kb_files(root_dir: str) -> List[str]:
    out: List[str] = []
    if not os.path.isdir(root_dir):
        return out
    for root, _, files in os.walk(root_dir):
        for f in files:
            ext = os.path.splitext(f)[1].lower()
            if ext in KB_ALLOWED_EXTENSIONS:
                out.append(os.path.join(root, f))
    return out


def build_kb_context(db_key: str, user_prompt: str, max_files: int = MAX_KB_CANDIDATE_FILES) -> Dict[str, Any]:
    paths = get_kb_paths(db_key)
    db_files = _iter_kb_files(paths["knowledge_base_db_dir"])
    shared_files = _iter_kb_files(paths["knowledge_base_shared_dir"])

    tokens = set(_tokenize_for_match(user_prompt))

    def score(path: str) -> tuple:
        base = os.path.basename(path).lower()
        name_score = sum(1 for t in tokens if t in base)
        db_boost = 1 if path.startswith(paths["knowledge_base_db_dir"]) else 0
        return (name_score, db_boost, base)

    ranked = sorted(db_files + shared_files, key=score, reverse=True)
    selected = ranked[:max_files]
    primary = selected[:5]

    return {
        **paths,
        "knowledge_base_primary_files": primary,
        "knowledge_base_candidate_files": selected,
    }


def normalize_model_output(text: str) -> str:
    text = (text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text.strip(), flags=re.I)
        text = re.sub(r"\s*```$", "", text.strip())
    return text.strip()


def extract_first_json_block(text: str) -> str:
    s = (text or "")
    starts = [i for i in [s.find("{"), s.find("[")] if i >= 0]
    if not starts:
        raise ValueError("No JSON found")
    start = min(starts)
    stack: List[str] = []
    in_str = False
    esc = False
    for i in range(start, len(s)):
        ch = s[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        if ch == '"':
            in_str = True
            continue
        if ch in "[{":
            stack.append(ch)
        elif ch in "]}":
            if not stack:
                continue
            op = stack[-1]
            if (op == "{" and ch == "}") or (op == "[" and ch == "]"):
                stack.pop()
                if not stack:
                    return s[start:i+1]
    raise ValueError("Unbalanced JSON block")


def parse_model(model_cls, text: str, label: str = "model_output"):
    normalized = normalize_model_output(text)
    if not normalized:
        raise ValueError(f"No output received for {label}")
    return model_cls.model_validate_json(normalized)


def _strip_sql_comments(sql: str) -> str:
    sql = re.sub(r"/\*.*?\*/", " ", sql, flags=re.S)
    sql = re.sub(r"--.*?$", " ", sql, flags=re.M)
    return sql


def check_select_only_sql(sql: str) -> Optional[str]:
    s = _strip_sql_comments(sql).strip()
    if not s:
        return "Empty SQL is not allowed"

    if ";" in s[:-1]:
        return "Only single-statement SQL is allowed"

    head = s.lower().lstrip("(")
    if not (head.startswith("select") or head.startswith("with")):
        return "SQL must start with SELECT or WITH"

    banned = re.compile(
        r"\b(insert|update|delete|merge|drop|alter|truncate|create|exec|execute|xp_cmdshell|sp_configure)\b",
        flags=re.I,
    )
    if banned.search(s):
        return "Unsafe SQL keyword detected"

    if re.search(r"\bselect\b[\s\S]*\binto\b", s, flags=re.I):
        return "SELECT INTO is not allowed"

    return None


def enforce_sql_safety_sqls(sqls: List[SQLQuery]):
    for q in sqls:
        reason = check_select_only_sql(q.sql)
        if reason:
            raise ValueError(f"Unsafe SQL in '{q.name}': {reason}")


def enforce_analyze_result_safety(ares: AnalyzeResult):
    enforce_sql_safety_sqls(ares.sql_appendix)
    if ares.detail_query:
        reason = check_select_only_sql(ares.detail_query.sql)
        if reason:
            raise ValueError(f"Unsafe detail SQL in '{ares.detail_query.name}': {reason}")


chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
_openai_key = os.getenv("OPENAI_API_KEY")
_embedding_fn = None
if _openai_key:
    _embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
        api_key=_openai_key,
        model_name=os.getenv("CHROMA_OPENAI_EMBED_MODEL", "text-embedding-3-small"),
    )


def get_collection(db_key: str):
    return chroma_client.get_or_create_collection(
        name=collection_name_for(db_key),
        embedding_function=_embedding_fn,
        metadata={"db_key": db_key},
    )


def rag_retrieve(db_key: str, query: str, k: int = 8, max_chars: int = 2200) -> str:
    col = get_collection(db_key)
    res = col.query(query_texts=[query], n_results=k)
    docs = (res.get("documents") or [[]])[0] or []
    return ("\n\n".join(docs[:k]))[:max_chars]


def rag_upsert_docs(db_key: str, items: List[Dict[str, Any]]):
    if not items:
        return
    col = get_collection(db_key)
    col.upsert(
        ids=[x["id"] for x in items],
        documents=[x["doc"] for x in items],
        metadatas=[x.get("meta", {}) for x in items],
    )


def rag_upsert_catalog(db_key: str, catalog: DBCatalogLite):
    items = []
    now = datetime.utcnow().isoformat()

    for entry in catalog.table_columns:
        t = entry.table
        cols = entry.columns
        items.append({
            "id": f"table::{t}",
            "doc": (
                f"TABLE {t}\n"
                f"Columns: {', '.join(c.col for c in cols[:120])}"
            ),
            "meta": {"type": "table", "table": t, "db_key": db_key, "updated_at": now},
        })

    for e in catalog.eav_hints:
        items.append({
            "id": f"eav::{e.table}",
            "doc": (
                f"EAV_HINT {e.table}\n"
                f"attribute_cols={e.attribute_cols}\n"
                f"value_cols={e.value_cols}"
            ),
            "meta": {"type": "eav_hint", "table": e.table, "db_key": db_key, "updated_at": now},
        })

    rag_upsert_docs(db_key, items)


def rag_upsert_semantics(db_key: str, user_request: str, ares: AnalyzeResult, catalog: Optional[DBCatalogLite] = None):
    now = datetime.utcnow().isoformat()
    items = []

    for q in ares.sql_appendix:
        items.append({
            "id": f"sql::{q.name}::{abs(hash(q.sql)) % (10**10)}",
            "doc": f"REQUEST: {user_request}\nQUERY_NAME: {q.name}\nSQL: {q.sql}",
            "meta": {"type": "sql_pattern", "db_key": db_key, "updated_at": now},
        })

    if ares.detail_query:
        dq = ares.detail_query
        items.append({
            "id": f"detail_sql::{dq.name}::{abs(hash(dq.sql)) % (10**10)}",
            "doc": f"REQUEST: {user_request}\nDETAIL_QUERY: {dq.name}\nSQL: {dq.sql}",
            "meta": {"type": "detail_sql", "db_key": db_key, "updated_at": now},
        })

    if catalog:
        for ct in catalog.candidate_fact_tables[:5]:
            items.append({
                "id": f"fact::{ct.name}",
                "doc": f"FACT_CANDIDATE {ct.name}; confidence={ct.confidence}; why={ct.why}",
                "meta": {"type": "fact_candidate", "table": ct.name, "db_key": db_key, "updated_at": now},
            })

    rag_upsert_docs(db_key, items)


def _extract_jsonish_from_tool_result(result: Any) -> Any:
    if not result or not getattr(result, "content", None):
        raise ValueError("Tool returned no content")

    # MCP results can contain text, structured json, or multiple parts.
    for part in result.content:
        # Some SDK objects expose .json/.data as methods.
        for attr in ("json", "data"):
            if not hasattr(part, attr):
                continue
            val = getattr(part, attr)
            if callable(val):
                try:
                    val = val()
                except TypeError:
                    # Method requires args; ignore and continue.
                    continue
            if val is not None and not callable(val):
                return val

        txt = getattr(part, "text", None)
        if txt and not callable(txt):
            norm = normalize_model_output(str(txt))
            if not norm:
                continue
            try:
                return json.loads(norm)
            except Exception:
                # Some servers return human text + JSON; keep scanning parts.
                continue

    # Fall back to the first text part for easier debugging.
    first = result.content[0] if result.content else None
    first_text = getattr(first, "text", "") if first is not None else ""
    if first_text and not callable(first_text):
        return str(first_text)
    raise ValueError("Tool content had no parseable JSON/data payload")


def _rows_from_result_obj(obj: Any):
    # Supports common mssql-mcp response shapes.
    if isinstance(obj, str):
        norm = normalize_model_output(obj)
        # Try direct JSON first.
        try:
            return _rows_from_result_obj(json.loads(norm))
        except Exception:
            pass
        # Try embedded JSON block next.
        try:
            return _rows_from_result_obj(json.loads(extract_first_json_block(norm)))
        except Exception:
            pass
        # Last resort for python-literal-like payloads.
        try:
            lit = ast.literal_eval(norm)
            return _rows_from_result_obj(lit)
        except Exception:
            preview = norm[:240].replace("\n", " ")
            raise ValueError(f"Unsupported execute_query text payload: {preview}")

    if isinstance(obj, list):
        if not obj:
            return [], []
        if isinstance(obj[0], dict):
            columns = list(obj[0].keys())
            rows = [[r.get(c) for c in columns] for r in obj]
            return columns, rows
        if isinstance(obj[0], list):
            return [], obj

    if isinstance(obj, dict):
        # Common MCP envelope: {"type":"text", "text":"{...}"}
        if isinstance(obj.get("text"), str):
            return _rows_from_result_obj(obj["text"])
        if isinstance(obj.get("result"), dict):
            return _rows_from_result_obj(obj["result"])
        if isinstance(obj.get("output"), dict):
            return _rows_from_result_obj(obj["output"])
        if isinstance(obj.get("data"), (dict, list)):
            return _rows_from_result_obj(obj["data"])
        if isinstance(obj.get("rows"), list) and isinstance(obj.get("columns"), list):
            cols_raw = obj["columns"]
            if cols_raw and isinstance(cols_raw[0], dict):
                columns = [str(c.get("name") or c.get("column") or c.get("field") or "") for c in cols_raw]
            else:
                columns = [str(c) for c in cols_raw]
            rows_raw = obj["rows"]
            if rows_raw and isinstance(rows_raw[0], dict):
                if not columns or all(c == "" for c in columns):
                    columns = list(rows_raw[0].keys())
                rows = [[r.get(c) for c in columns] for r in rows_raw]
            else:
                rows = rows_raw
            return columns, rows
        if isinstance(obj.get("records"), list):
            return _rows_from_result_obj(obj["records"])
        if isinstance(obj.get("items"), list):
            return _rows_from_result_obj(obj["items"])
        if isinstance(obj.get("recordset"), list):
            return _rows_from_result_obj(obj["recordset"])
        if isinstance(obj.get("resultset"), list):
            return _rows_from_result_obj(obj["resultset"])
        if isinstance(obj.get("results"), list) and obj["results"]:
            return _rows_from_result_obj(obj["results"][0])

    raise ValueError(f"Unsupported execute_query result format for export: {type(obj)}")


async def execute_query_programmatically(sql: str) -> tuple[List[str], List[List[Any]]]:
    reason = check_select_only_sql(sql)
    if reason:
        raise ValueError(f"Unsafe programmatic SQL: {reason}")

    # mssql-mcp variants use either 'sql' or 'query'.
    try:
        result = await mssql_server.call_tool("execute_query", {"sql": sql})
    except Exception:
        result = await mssql_server.call_tool("execute_query", {"query": sql})

    obj = _extract_jsonish_from_tool_result(result)
    columns, rows = _rows_from_result_obj(obj)
    return columns, rows


def _build_pyodbc_connection_string() -> str:
    # Prefer explicit ODBC/DSN connection string when already provided.
    raw = (os.getenv("MSSQL_CONNECTION_STRING") or "").strip()
    if raw and ("driver=" in raw.lower() or "dsn=" in raw.lower()):
        return raw

    host = (os.getenv("MSSQL_HOST") or "").strip()
    port = (os.getenv("MSSQL_PORT") or "1433").strip()
    database = (os.getenv("MSSQL_DATABASE") or "").strip()
    user = (os.getenv("MSSQL_USER") or "").strip()
    password = os.getenv("MSSQL_PASSWORD") or ""
    encrypt_raw = (os.getenv("MSSQL_ENCRYPT") or "true").strip().lower()
    encrypt = "yes" if encrypt_raw in {"1", "true", "yes", "y", "on"} else "no"

    missing = [k for k, v in {
        "MSSQL_HOST": host,
        "MSSQL_DATABASE": database,
        "MSSQL_USER": user,
        "MSSQL_PASSWORD": password,
    }.items() if not v]
    if missing:
        raise RuntimeError(f"Missing DB env vars for pyodbc: {', '.join(missing)}")

    trust = "no" if encrypt == "yes" else "yes"
    return (
        "Driver={ODBC Driver 18 for SQL Server};"
        f"Server=tcp:{host},{port};"
        f"Database={database};"
        f"Uid={user};"
        f"Pwd={password};"
        f"Encrypt={encrypt};"
        f"TrustServerCertificate={trust};"
    )


def _execute_query_pyodbc_sync(sql: str) -> tuple[List[str], List[List[Any]]]:
    if pyodbc is None:
        raise RuntimeError("pyodbc is not installed in this environment")
    conn_str = _build_pyodbc_connection_string()

    conn = pyodbc.connect(conn_str)
    try:
        cur = conn.cursor()
        cur.execute(sql)
        columns = [d[0] for d in (cur.description or [])]
        rows = [list(r) for r in cur.fetchall()]
        return columns, rows
    finally:
        conn.close()


async def execute_query_programmatically_large(sql: str) -> tuple[List[str], List[List[Any]]]:
    reason = check_select_only_sql(sql)
    if reason:
        raise ValueError(f"Unsafe programmatic SQL: {reason}")
    return await asyncio.to_thread(_execute_query_pyodbc_sync, sql)


def write_rows_to_csv(columns: List[str], rows: List[List[Any]], prefix: str = "ai_analytics_detail") -> str:
    fd, path = tempfile.mkstemp(prefix=prefix, suffix=".csv")
    os.close(fd)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(columns)
        w.writerows(rows)
    return path


def read_top_rows_from_csv(path: str, n: int = 10) -> tuple[List[str], List[List[str]]]:
    with open(path, "r", encoding="utf-8", newline="") as f:
        r = csv.reader(f)
        all_rows = list(r)
    if not all_rows:
        return [], []
    return all_rows[0], all_rows[1:1+n]


In [7]:

PLANNER_INSTRUCTIONS = """
You are Planner for an AI analytics workflow.

Goal:
- decide if schema discovery is needed

Input:
- user_request
- rag_context (optional)
- knowledge_base_db_dir (optional)
- knowledge_base_shared_dir (optional)
- knowledge_base_primary_files (optional)
- knowledge_base_candidate_files (optional)

Output JSON:
{
  "require_schema_refresh": true/false,
  "rationale": "short reason"
}

Rules:
- require_schema_refresh=true when rag_context is empty/weak or request uses unfamiliar schema terms.
- Do not call document tools in Planner.
- Use knowledge_base_primary_files and knowledge_base_candidate_files (filename relevance only) for planning decisions.
- Prefer searching knowledge_base_db_dir first, then knowledge_base_shared_dir.
- Use only knowledge_base_primary_files and knowledge_base_candidate_files for document selection.
- Do not run directory scanning tool calls against knowledge base paths.
"""


SQL_ANALYST_INSTRUCTIONS = """
You are an expert SQL Analyst. You handle either DISCOVER or ANALYZE mode.
You are responsible for BOTH analysis and self-verification.

Allowed tools:
- execute_query
- list_tables
- describe_table
- sample_data
- document content-read tools only:
  - read_pdf (PDF)
  - get_document_text (Word)
  - read_data_from_excel (Excel)

Input JSON:
- task_mode: DISCOVER | ANALYZE
- user_request
- rag_context (optional)
- knowledge_base_db_dir (optional)
- knowledge_base_shared_dir (optional)
- knowledge_base_primary_files (optional)
- knowledge_base_candidate_files (optional)
- catalog (optional, ANALYZE mode)
- verifier_fixes (optional, ANALYZE mode)
- small_row_threshold (optional, default 1000)
- max_direct_columns (optional, default 12)

Global rules:
- Work autonomously.
- Do not claim data unavailable before at least one relevant attempt.
- SQL safety: SELECT/WITH only; never emit DDL/DML/EXEC patterns.
- For latest/most recent logic, compute explicitly via MAX(date_col).
- If knowledge_base_primary_files is non-empty, call at least one document read tool before SQL exploration.
- Prefer knowledge_base_primary_files first, then knowledge_base_candidate_files.
- Use only those provided lists and read only relevant files from them.
- Never call directory/listing/metadata/outline/validation document tools.
- Explicitly forbidden examples: list_available_documents, get_document_info, get_document_outline, get_document_xml, get_workbook_metadata, validate_excel_range, validate_formula_syntax.
- Never call document creation/editing/formatting tools.

DISCOVER mode (strict):
1) Run INFORMATION_SCHEMA.TABLES once.
2) Run INFORMATION_SCHEMA.COLUMNS once.
3) Use targeted subset only for describe_table.
4) Use up to 2 SELECT TOP 1 checks if still ambiguous.
5) Return DBCatalogLite immediately.

DISCOVER output caps (mandatory):
- candidate_fact_tables: at most 6
- table_columns: at most 8 tables
- columns per table entry: at most 20
- eav_hints: at most 3
- notes: at most 8

ANALYZE mode (strict):
You MUST follow count-first with dual-query strategy:
1) Build base query logic for the user request.
   - Use only columns confirmed in catalog.table_columns, describe_table, or INFORMATION_SCHEMA.COLUMNS.
   - Never assume common column names (for example ModifiedDate/UpdatedDate) unless explicitly confirmed.
2) Build and execute count_sql first:
   - SELECT COUNT(*) AS total_rows FROM ( <base query without ORDER BY if needed> ) q
3) Decide mode:
   - SMALL_DIRECT if total_rows <= small_row_threshold and projected columns <= max_direct_columns.
   - LARGE_SPLIT otherwise.
4) If SMALL_DIRECT:
   - Execute a direct query suitable for response.
   - Return insight_tables from direct query.
   - detail_query = null.
5) If LARGE_SPLIT:
   - Build and execute one aggregated insight query for LLM-friendly results.
   - Build a detail_query that satisfies full-detail requirement.
   - Validate detail_query compiles by running a TOP 1 (or TOP 0) dry-run version before returning.
   - DO NOT execute detail_query.
   - detail_query must be a single SELECT statement only (no DECLARE, temp tables, variables, or multi-statement batches).
6) Return AnalyzeResult JSON only.

AnalyzeResult requirements:
- include mode, count_sql, total_rows
- sql_appendix must include exact SQL for count + executed insight SQL
- detail_query must be present if LARGE_SPLIT
- include assumptions/caveats when proxies/ambiguity are used
- Every SQL string returned (count_sql, sql_appendix, detail_query) must be exactly one statement.
"""


CHART_MAPPER_INSTRUCTIONS = """
You are Chart Mapper. You may inspect chart MCP tools to choose correct mappings, but do not execute chart tools.

Input JSON includes:
- user_request
- columns
- sample_rows (top 10 rows)

Task:
- Determine whether charting is applicable.
- If applicable, return mapping fields for chart construction.

Return JSON matching ChartMapping.
Rules:
- You may inspect available chart tools/schemas before mapping.
- Do not call chart generation tools from this agent.
- Use chart_type in {line,column,bar,pie}
- For line/column/bar: provide x_field and y_field.
- For pie: provide category_field and value_field.
- If not applicable, set applicable=false and explain in reason.
"""


NARRATOR_INSTRUCTIONS = """
You are Narrator. No DB tools.
Allowed document tools (read-only): read_pdf, get_document_text, read_data_from_excel.

Input JSON includes:
- mode
- total_rows
- insight_tables
- assumptions
- caveats
- sql_appendix
- detail_query (optional)
- detail_export_path (optional)
- chart_artifact (optional)
- knowledge_base_db_dir (optional)
- knowledge_base_shared_dir (optional)
- knowledge_base_primary_files (optional)
- knowledge_base_candidate_files (optional)

Rules:
- If knowledge_base_primary_files is non-empty, call at least one document read tool before generating final narrative.
- Prefer knowledge_base_primary_files first, then knowledge_base_candidate_files.
- Use only those provided lists for document selection.
- Do not run directory scanning tool calls against knowledge base paths.
- Do not call document metadata/outline/validation/listing or editing/creation tools.

Output JSON matching NarratorOutput with fields:
- user_output
- developer_output

user_output rules (strict):
- Relevant and business-friendly with simple english.
- focus on outcomes and insights only, include key insights, trends if applicable.
- do NOT mention SQL, query mechanics, tables, rows, columns, joins, schemas, or other database internals
- if chart_artifact.image_url exists, include a Chart section with markdown image

developer_output rules:
- include Data Quality Notes
- include Technical Notes
- include SQL Appendix (exact SQL)
- include Detail Export path when provided
"""


CHAT_ORCHESTRATOR_INSTRUCTIONS = """
You are the user-facing chat agent for AI Analytics.

You MUST use tools to perform data analysis.

Primary tool:
- run_analytics_pipeline: use for normal user questions.

Optional tool:
- run_structured_analysis: use only when user explicitly requests technical/debug raw output.

Rules:
- Keep a single consistent conversational voice and use simple english.
- Do not invent data; rely on tool outputs.
- For now, call run_analytics_pipeline exactly once and return its output verbatim.
- The response must include both User Output and Developer Output sections.
"""


In [8]:
# Legacy notes removed


In [9]:

# =========================
# BUILD AGENTS (SIMPLIFIED)
# =========================


def build_agents(db_key: str):
    default_settings = ModelSettings(temperature=DEFAULT_AGENT_TEMPERATURE)
    sql_analyst_settings = ModelSettings(temperature=SQL_ANALYST_TEMPERATURE)

    planner = Agent(
        name="Planner",
        model=MODEL_PLANNER,
        model_settings=default_settings,
        instructions=PLANNER_INSTRUCTIONS,
        output_type=PlannerDecision,
        mcp_servers=[],
    )

    sql_analyst_discover = Agent(
        name="SQL Analyst Discover",
        model=MODEL_GENERAL,
        model_settings=sql_analyst_settings,
        instructions=SQL_ANALYST_INSTRUCTIONS,
        output_type=DBCatalogLite,
        mcp_servers=[mssql_server, pdf_reader_server, word_reader_server, excel_server],
    )

    sql_analyst_query = Agent(
        name="SQL Analyst Query",
        model=MODEL_GENERAL,
        model_settings=sql_analyst_settings,
        instructions=SQL_ANALYST_INSTRUCTIONS,
        output_type=AnalyzeResult,
        mcp_servers=[mssql_server, pdf_reader_server, word_reader_server, excel_server],
    )

    chart_mapper = Agent(
        name="Chart Mapper",
        model=MODEL_GENERAL,
        model_settings=default_settings,
        instructions=CHART_MAPPER_INSTRUCTIONS,
        output_type=ChartMapping,
        mcp_servers=[chart_server],
    )

    narrator = Agent(
        name="Narrator",
        model=MODEL_GENERAL,
        model_settings=default_settings,
        instructions=NARRATOR_INSTRUCTIONS,
        output_type=NarratorOutput,
        mcp_servers=[pdf_reader_server, word_reader_server, excel_server],
    )

    return {
        "planner": planner,
        "sql_analyst_discover": sql_analyst_discover,
        "sql_analyst_query": sql_analyst_query,
        "chart_mapper": chart_mapper,
        "narrator": narrator,
    }


In [10]:

# =========================
# END-TO-END PIPELINE
# =========================


async def run_structured(
    agent: Agent,
    payload: Dict[str, Any],
    model_cls,
    *,
    debug_label: str,
    max_turns: int,
):
    res = await Runner.run(agent, json.dumps(payload), max_turns=max_turns)
    out = res.final_output

    if isinstance(out, model_cls):
        return out
    if isinstance(out, dict):
        return model_cls.model_validate(out)
    if isinstance(out, str):
        return parse_model(model_cls, out, label=debug_label)
    raise ValueError(f"{debug_label}: unexpected output type {type(out)}")


async def run_planner(agents: Dict[str, Agent], db_key: str, user_prompt: str, rag_ctx: str) -> PlannerDecision:
    kb_ctx = build_kb_context(db_key, user_prompt)
    payload = {
        "user_request": user_prompt,
        "rag_context": rag_ctx,
        **kb_ctx,
    }
    return await run_structured(
        agents["planner"],
        payload,
        PlannerDecision,
        debug_label="Planner",
        max_turns=MAX_TURNS_PLANNER,
    )


async def run_sql_analyst_discover(agents: Dict[str, Agent], db_key: str, user_prompt: str, rag_ctx: str) -> DBCatalogLite:
    kb_ctx = build_kb_context(db_key, user_prompt)
    payload = {
        "task_mode": "DISCOVER",
        "user_request": user_prompt,
        "rag_context": rag_ctx,
        **kb_ctx,
    }
    catalog = await run_structured(
        agents["sql_analyst_discover"],
        payload,
        DBCatalogLite,
        debug_label="SQL Analyst DISCOVER",
        max_turns=MAX_TURNS_SQL_ANALYST_DISCOVER,
    )
    rag_upsert_catalog(db_key, catalog)
    return catalog


async def run_sql_analyst_query(
    agents: Dict[str, Agent],
    db_key: str,
    user_prompt: str,
    rag_ctx: str,
    catalog: Optional[DBCatalogLite],
    verifier_fixes: Optional[List[str]] = None,
) -> AnalyzeResult:
    kb_ctx = build_kb_context(db_key, user_prompt)
    payload = {
        "task_mode": "ANALYZE",
        "user_request": user_prompt,
        "rag_context": rag_ctx,
        "catalog": catalog.model_dump() if catalog else None,
        "verifier_fixes": verifier_fixes or [],
        "small_row_threshold": SMALL_ROW_THRESHOLD,
        "max_direct_columns": MAX_DIRECT_COLUMNS,
        **kb_ctx,
    }
    ares = await run_structured(
        agents["sql_analyst_query"],
        payload,
        AnalyzeResult,
        debug_label="SQL Analyst ANALYZE",
        max_turns=MAX_TURNS_SQL_ANALYST_QUERY,
    )
    enforce_analyze_result_safety(ares)
    return ares


async def run_chart_mapper(agents: Dict[str, Agent], user_prompt: str, columns: List[str], sample_rows: List[List[Any]]) -> ChartMapping:
    payload = {
        "user_request": user_prompt,
        "columns": columns,
        "sample_rows": sample_rows[:10],
    }
    return await run_structured(
        agents["chart_mapper"],
        payload,
        ChartMapping,
        debug_label="Chart Mapper",
        max_turns=MAX_TURNS_CHART_MAPPER,
    )


def _coerce_float(v: Any) -> Optional[float]:
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).strip().replace(",", "")
    try:
        return float(s)
    except Exception:
        return None


def _numeric_score(rows: List[List[Any]], col_i: int, sample_n: int = 30) -> int:
    score = 0
    for r in rows[:sample_n]:
        if len(r) <= col_i:
            continue
        if _coerce_float(r[col_i]) is not None:
            score += 1
    return score


def _to_chart_data(rows: List[List[Any]], columns: List[str], mapping: ChartMapping) -> Dict[str, Any]:
    idx = {c: i for i, c in enumerate(columns)}

    if mapping.chart_type in ["line", "column", "bar"]:
        if not mapping.x_field or not mapping.y_field:
            raise ValueError("Chart mapping missing x_field/y_field")
        xi = idx[mapping.x_field]
        yi = idx[mapping.y_field]

        # Auto-correct mapper mistakes: y should be numeric.
        if _numeric_score(rows, yi) == 0 and _numeric_score(rows, xi) > 0:
            xi, yi = yi, xi

        data = []
        for r in rows:
            if len(r) <= max(xi, yi):
                continue
            val = _coerce_float(r[yi])
            if val is None:
                continue
            if mapping.chart_type == "line":
                data.append({"time": str(r[xi]), "value": val})
            else:
                data.append({"category": str(r[xi]), "value": val})

        if not data:
            raise ValueError("No numeric values available for chart y-axis")

        if mapping.chart_type == "line":
            return {
                "tool": "generate_line_chart",
                "input": {
                    "data": data,
                    "xField": "time",
                    "yField": "value",
                    "title": "Line chart",
                },
            }
        if mapping.chart_type == "bar":
            return {
                "tool": "generate_bar_chart",
                "input": {
                    "data": data,
                    "title": "Bar chart",
                },
            }
        return {
            "tool": "generate_column_chart",
            "input": {
                "data": data,
                "title": "Column chart",
            },
        }

    if mapping.chart_type == "pie":
        if not mapping.category_field or not mapping.value_field:
            raise ValueError("Pie mapping missing category/value fields")
        ci = idx[mapping.category_field]
        vi = idx[mapping.value_field]
        data = []
        for r in rows:
            if len(r) <= max(ci, vi):
                continue
            val = _coerce_float(r[vi])
            if val is None:
                continue
            data.append({"category": str(r[ci]), "value": val})
        if not data:
            raise ValueError("No numeric values available for pie chart")
        return {
            "tool": "generate_pie_chart",
            "input": {
                "data": data,
                "angleField": "value",
                "colorField": "category",
                "title": "Pie chart",
            },
        }

    raise ValueError(f"Unsupported chart type: {mapping.chart_type}")


def _extract_chart_image_url(result: Any) -> Optional[str]:
    if not result or not getattr(result, "content", None):
        return None
    for part in result.content:
        txt = getattr(part, "text", None)
        if txt and isinstance(txt, str):
            s = txt.strip()
            if s.startswith("http"):
                return s
            try:
                obj = json.loads(normalize_model_output(s))
                if isinstance(obj, dict):
                    for k in ["image_url", "url", "chartUrl"]:
                        if isinstance(obj.get(k), str):
                            return obj[k]
            except Exception:
                pass
    return None


async def maybe_generate_chart(
    agents: Dict[str, Agent],
    user_prompt: str,
    ares: AnalyzeResult,
    include_chart: bool,
    detail_export_path: Optional[str] = None,
) -> Optional[Dict[str, Any]]:
    if not include_chart or not ares.insight_tables:
        return None

    t = ares.insight_tables[0]
    if len(t.columns) < 2 or not t.rows:
        return None

    mapping = await run_chart_mapper(agents, user_prompt, t.columns, t.rows[:10])
    if not mapping.applicable:
        return None

    if detail_export_path:
        columns, top_rows = read_top_rows_from_csv(detail_export_path, n=2000)
        chart_source_cols = columns
        chart_source_rows = top_rows
    else:
        chart_source_cols = t.columns
        chart_source_rows = t.rows[:2000]

    if not chart_source_cols or not chart_source_rows:
        return None

    try:
        chart_cfg = _to_chart_data(chart_source_rows, chart_source_cols, mapping)
    except Exception as e:
        print(f"Chart mapping skipped: {e}")
        return None

    result = await chart_server.call_tool(chart_cfg["tool"], chart_cfg["input"])
    image_url = _extract_chart_image_url(result)
    if image_url:
        return {
            "image_url": image_url,
            "mapping": mapping.model_dump(),
            "source": detail_export_path or "insight_table",
        }
    return None


async def run_narrator(
    agents: Dict[str, Agent],
    db_key: str,
    user_prompt: str,
    ares: AnalyzeResult,
    chart_artifact: Optional[Dict[str, Any]],
    detail_export_path: Optional[str],
) -> NarratorOutput:
    kb_ctx = build_kb_context(db_key, user_prompt)
    payload = {
        "mode": ares.mode,
        "total_rows": ares.total_rows,
        "insight_tables": [t.model_dump() for t in ares.insight_tables],
        "assumptions": ares.assumptions,
        "caveats": ares.caveats,
        "sql_appendix": [q.model_dump() for q in ares.sql_appendix] + ([ares.detail_query.model_dump()] if ares.detail_query else []),
        "detail_query": ares.detail_query.model_dump() if ares.detail_query else None,
        "detail_export_path": detail_export_path,
        "chart_artifact": chart_artifact,
        **kb_ctx,
    }
    return await run_structured(
        agents["narrator"],
        payload,
        NarratorOutput,
        debug_label="Narrator",
        max_turns=MAX_TURNS_NARRATOR,
    )


async def run_request(user_prompt: str, db_key: str, use_narrator: bool = True) -> NotebookRunResult:
    agents = build_agents(db_key)

    rag_ctx = rag_retrieve(db_key, user_prompt, k=8, max_chars=5000)
    plan = await run_planner(agents, db_key, user_prompt, rag_ctx)

    catalog = None
    # if plan.require_schema_refresh or not rag_ctx.strip():
    #     print("Discovering")
    #     catalog = await run_sql_analyst_discover(agents, db_key, user_prompt, rag_ctx)
    #     rag_ctx = rag_retrieve(db_key, user_prompt, k=10, max_chars=5000)

    print("Analyzing")
    ares = await run_sql_analyst_query(agents, db_key, user_prompt, rag_ctx, catalog)

    detail_export_path = None
    if ares.mode == "LARGE_SPLIT" and ares.detail_query:
        print("LARGE Split")
        print(f"Total rows: {ares.total_rows}")
        cols, rows = await execute_query_programmatically_large(ares.detail_query.sql)
        detail_export_path = write_rows_to_csv(cols, rows)

    rag_upsert_semantics(db_key, user_prompt, ares, catalog=catalog)

    chart_artifact = await maybe_generate_chart(
        agents,
        user_prompt,
        ares,
        include_chart=True,
        detail_export_path=detail_export_path,
    )

    if chart_artifact is None:
        print("NO chart.")

    print("Narrating")

    user_output = None
    developer_output = None
    if use_narrator:
        nout = await run_narrator(agents, db_key, user_prompt, ares, chart_artifact, detail_export_path)
        user_output = nout.user_output
        developer_output = nout.developer_output
        final_output = "\n\n---\n\n".join([
            "# User Output\n" + user_output,
            "# Developer Output\n" + developer_output,
        ])
    else:
        final_output = json.dumps(ares.model_dump(), indent=2)

    return NotebookRunResult(
        final_output=final_output,
        user_output=user_output,
        developer_output=developer_output,
        verifier_verdict="PASS",
        verification_reasons=[],
    )



def build_chat_agent(db_key: str) -> Agent:
    @function_tool
    async def run_analytics_pipeline(user_request: str) -> str:
        """Run the full analytics backend pipeline and return combined user+developer output."""
        result = await run_request(user_request, db_key=db_key, use_narrator=True)
        return result.final_output

    @function_tool
    async def run_structured_analysis(user_request: str) -> str:
        """Run backend analysis without narration and return raw structured output."""
        result = await run_request(user_request, db_key=db_key, use_narrator=False)
        return result.final_output

    return Agent(
        name="Chat Orchestrator",
        model=MODEL_GENERAL,
        model_settings=ModelSettings(temperature=DEFAULT_AGENT_TEMPERATURE),
        instructions=CHAT_ORCHESTRATOR_INSTRUCTIONS,
        tools=[run_analytics_pipeline, run_structured_analysis],
        mcp_servers=[],
    )


async def run_chat_request(user_prompt: str, db_key: str) -> str:
    chat_agent = build_chat_agent(db_key)
    res = await Runner.run(chat_agent, user_prompt, max_turns=MAX_TURNS_CHAT_ORCHESTRATOR)
    out = res.final_output
    if isinstance(out, str):
        return out
    if isinstance(out, dict):
        return json.dumps(out, indent=2)
    return str(out)


In [11]:

# =========================
# SMALL TEST BLOCKS
# =========================


async def test_planner(db_key: str, prompt: str):
    agents = build_agents(db_key)
    rag_ctx = rag_retrieve(db_key, prompt)
    return await run_planner(agents, db_key, prompt, rag_ctx)


async def test_discovery(db_key: str, prompt: str):
    agents = build_agents(db_key)
    rag_ctx = rag_retrieve(db_key, prompt)
    return await run_sql_analyst_discover(agents, db_key, prompt, rag_ctx)


async def test_analysis(db_key: str, prompt: str, catalog: Optional[DBCatalogLite] = None):
    agents = build_agents(db_key)
    rag_ctx = rag_retrieve(db_key, prompt)
    return await run_sql_analyst_query(agents, db_key, prompt, rag_ctx, catalog)


async def test_chart_mapper(prompt: str, columns: List[str], sample_rows: List[List[Any]]):
    agents = build_agents("dev")
    return await run_chart_mapper(agents, prompt, columns, sample_rows)



async def test_chat(db_key: str, prompt: str):
    return await run_chat_request(prompt, db_key)


In [12]:
# =========================
# EXAMPLE USAGE
# =========================

DB_KEY = "mssql_demo"
USER_PROMPT1 = (
    "Find the latest CreationDate in dbo.CompletedChecklist, then show daily completion counts "
    "for the 30 days ending on that latest date. Generate the most appropriate chart if applicable. "
    "Only generate one chart."
)

USER_PROMPT2 = (
    "Can you give me a representation of how the latest questionare /checklist form woudl look for say this checklist Riegelwood Mill - Safety Observation"
)

USER_PROMPT = (
    "Which OCR checklist from what sites had the most negative incidents for the first month of 2025, "
    "use the following tables and more [dbo].[Checklist], [dbo].[ChecklistStructure], [dbo].[CompletedChecklist], "
    "[dbo].[CompletedChecklist_Answer], Try other related tables. i think it might be in a Entity-Attribute-Value (EAV) "
)

# Chat orchestrator path (user-facing backend)
async with mssql_server, chart_server, pdf_reader_server, word_reader_server, excel_server:
    chat_result = await run_chat_request(USER_PROMPT1, DB_KEY)
    display(Markdown(chat_result))


Analyzing


/tmp/ipykernel_18725/686369234.py:237: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow().isoformat()


Narrating


# User Output
## Daily checklist completions (last 30 days ending on the latest activity)

### What’s happening
- Completion volume is **highly uneven day-to-day**, with several **large spike days** and a number of **very low-activity days**.
- The biggest surges occur on:
  - **Jan 13: 105 completions**
  - **Jan 27: 101 completions**
  - **Jan 20: 80 completions**
- Outside of those surges, most active days sit in a more typical band of roughly **mid-teens to mid-30s** (e.g., 14–36).

### What this likely means (business interpretation)
- Work is probably being completed in **batches** (e.g., scheduled runs, weekly operational cycles, catch-up days, or centralized processing) rather than at a steady daily rate.
- The very low days (e.g., **1–2 completions**) suggest **non-working days**, **limited field activity**, or **process dependency** (e.g., completions only happen when a specific team is active).

### Suggested follow-ups
- Confirm whether spike days align with **scheduled inspections/audits**, **shift patterns**, or **end-of-period catch-up**.
- If steadier throughput is desired, consider operational changes to reduce batching (capacity planning, reminders, or staggered schedules).

## Chart

![Daily completion trend](https://mdn.alipayobjects.com/one_clip/afts/img/AWZ-SoQnU6EAAAAAUVAAAAgAoEACAQFr/original)

---

# Developer Output
## Data Quality Notes
- The daily series **only includes days with at least 1 completion**; days with zero completions are not present, which can make the trend look more volatile than it truly is.
- The execution layer applied **TOP(20)**, so only the **first 20 (earliest) days** are shown in the insight table even though `total_rows = 26`.
- Date handling assumes an **inclusive 30-day window** defined as: `[latest_date - 29 days, latest_date + 1 day)` based on the latest available completion timestamp.

## Technical Notes
- Metric: count of checklist completions grouped by completion date.
- Window anchor: latest completion timestamp in the source.
- Grouping: calendar date derived from the completion timestamp.

## SQL Appendix
### count_sql
```sql
SELECT COUNT(*) AS total_rows
FROM (
  SELECT
    CAST(cc.CreationDate AS date) AS CompletionDate,
    COUNT_BIG(*) AS CompletionCount
  FROM dbo.CompletedChecklist AS cc
  WHERE cc.CreationDate >= DATEADD(day, -29, (SELECT MAX(CreationDate) FROM dbo.CompletedChecklist))
    AND cc.CreationDate <  DATEADD(day,  1, (SELECT MAX(CreationDate) FROM dbo.CompletedChecklist))
  GROUP BY CAST(cc.CreationDate AS date)
) q;
```

### insight_sql_daily_counts_30d
```sql
SELECT
  CAST(cc.CreationDate AS date) AS CompletionDate,
  COUNT_BIG(*) AS CompletionCount
FROM dbo.CompletedChecklist AS cc
WHERE cc.CreationDate >= DATEADD(day, -29, (SELECT MAX(CreationDate) FROM dbo.CompletedChecklist))
  AND cc.CreationDate <  DATEADD(day,  1, (SELECT MAX(CreationDate) FROM dbo.CompletedChecklist))
GROUP BY CAST(cc.CreationDate AS date)
ORDER BY CompletionDate;
```

In [13]:
# 1) Inspect Chroma collection stats + sample records
DB_KEY = "mssql_demo"
col = get_collection(DB_KEY)

print("Collection:", col.name)
print("Count:", col.count())

peek = col.peek(limit=10)  # ids, documents, metadatas
for i, _id in enumerate(peek.get("ids", []), 1):
    doc = (peek.get("documents", [""] * i)[i-1] or "")[:1000]
    meta = (peek.get("metadatas", [{}] * i)[i-1] or {})
    print(f"\n{i}. id={_id}")
    print("meta:", meta)
    print("doc:", doc)


Collection: ai_analytics__mssql_demo
Count: 20

1. id=sql::count_sql::9940451884
meta: {'type': 'sql_pattern', 'updated_at': '2026-02-15T08:13:10.963407', 'db_key': 'mssql_demo'}
doc: REQUEST: Find the latest CreationDate in dbo.CompletedChecklist, then show daily completion counts for the 30 days ending on that latest date. Generate the most appropriate chart if applicable. Only generate one chart.
QUERY_NAME: count_sql
SQL: SELECT COUNT(*) AS total_rows FROM (SELECT CAST(cc.CreationDate AS date) AS CompletionDate, COUNT_BIG(*) AS CompletionCount FROM dbo.CompletedChecklist AS cc WHERE cc.CreationDate >= DATEADD(day, -29, (SELECT MAX(CreationDate) FROM dbo.CompletedChecklist)) AND cc.CreationDate < DATEADD(day, 1, (SELECT MAX(CreationDate) FROM dbo.CompletedChecklist)) GROUP BY CAST(cc.CreationDate AS date)) q;

2. id=sql::insight_sql_daily_counts_30d::3994156239
meta: {'type': 'sql_pattern', 'updated_at': '2026-02-15T08:13:10.963407', 'db_key': 'mssql_demo'}
doc: REQUEST: Find the la

In [14]:
# 2) Search memory for a prompt
q = "negative incidents OCR checklist site January 2025"
res = col.query(query_texts=[q], n_results=8)
for i, doc in enumerate((res.get("documents") or [[]])[0], 1):
    meta = ((res.get("metadatas") or [[]])[0][i-1]) if (res.get("metadatas") or [[]])[0] else {}
    print(f"\n{i}. meta={meta}")
    print(doc[:500])



1. meta={'updated_at': '2026-02-15T08:13:10.963407', 'db_key': 'mssql_demo', 'type': 'sql_pattern'}
REQUEST: Find the latest CreationDate in dbo.CompletedChecklist, then show daily completion counts for the 30 days ending on that latest date. Generate the most appropriate chart if applicable. Only generate one chart.
QUERY_NAME: count_sql
SQL: SELECT COUNT(*) AS total_rows FROM (SELECT CAST(cc.CreationDate AS date) AS CompletionDate, COUNT_BIG(*) AS CompletionCount FROM dbo.CompletedChecklist AS cc WHERE cc.CreationDate >= DATEADD(day, -29, (SELECT MAX(CreationDate) FROM dbo.CompletedChecklist

2. meta={'db_key': 'mssql_demo', 'updated_at': '2026-02-15T08:38:58.010527', 'type': 'sql_pattern'}
REQUEST: Find the latest CreationDate in dbo.CompletedChecklist, then show daily completion counts for the 30 days ending on that latest date. Generate the most appropriate chart if applicable. Only generate one chart.
QUERY_NAME: count_sql
SQL: SELECT COUNT(*) AS total_rows FROM (SELECT CAST(cc.

In [15]:
import os
from pathlib import Path

print("cwd:", os.getcwd())
print("CHROMA_DIR:", CHROMA_DIR)
print("CHROMA_DIR abs:", Path(CHROMA_DIR).resolve())

for c in chroma_client.list_collections():
    print(c.name, "count=", c.count())


cwd: /home/preshen/Projects/ai_analytics
CHROMA_DIR: /tmp/chroma_ai_analytics
CHROMA_DIR abs: /tmp/chroma_ai_analytics
ai_analytics__mssql_demo count= 20
